# LifeMate fine-tune LLM (Colab)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# Option A: unzip the code package uploaded to Drive (zip from ops/scripts/pack_code.sh).
!mkdir -p /content/lifemate && unzip -oq "$(ls -t /content/drive/MyDrive/zac2026/lifemate-code-*.zip | head -1)" -d /content/lifemate
# Option B: !git clone <repo-url> /content/lifemate
%cd /content/lifemate

In [ ]:
%pip install -q -r ops/requirements/notebook.txt -r ops/requirements/train.txt pytest
# vllm is only needed for the agent-eval stage (llm pipeline "eval"); pinned from uv.lock (0.32.0).
!nvidia-smi > /dev/null 2>&1 && pip install -q vllm==0.32.0 || echo "no GPU: skipping vllm"
# ASR engine for the ASR cache below (configs/v0.toml uses faster-whisper large-v3)
%pip install -q faster-whisper==1.2.1


In [ ]:
!python -c "from pathlib import Path; from lifemate.core.envinfo import detect; print(detect(Path('.')))"

In [ ]:
!python -m pytest -q tests/train -m "not data and not tokenizer"

In [ ]:
# Drive is the work dir (/content/drive/MyDrive/zac2026/runs/train/<name>), so a second session resumes by itself.
!ls -la /content/drive/MyDrive/zac2026/runs/train 2>/dev/null || echo "no previous training on Drive"

In [ ]:
# ASR cache (SPEC 6.2.1): configs/train/llm_v1.toml trains on 70% ASR text, read from data/asr/train_asr.json.
# Built once (slow: ASR on every train/val sample); reused from a restored runs folder when present.
import os, shutil, subprocess
cache = "data/asr/train_asr.json"
saved = None
if saved and os.path.isfile(saved) and not os.path.isfile(cache):
    os.makedirs("data/asr", exist_ok=True); shutil.copy(saved, cache); shutil.copy(saved + ".meta.json", cache + ".meta.json")
if not os.path.isfile(cache):
    subprocess.run(["python", "-m", "lifemate.train.asr_cache", "--config", "configs/v0.toml", "--env", "colab",
                    "--out", cache, "--ids", "configs/splits/train.txt", "configs/splits/val_seen.txt",
                    "configs/splits/val_unseen.txt"], check=True)
    if saved:
        os.makedirs(os.path.dirname(saved), exist_ok=True); shutil.copy(cache, saved); shutil.copy(cache + ".meta.json", saved + ".meta.json")
print("ASR cache:", cache, os.path.getsize(cache), "bytes")


In [ ]:
# One command (SPEC 6.2.6): data -> train -> select -> merge -> eval -> choose -> register.
# Add --from-stage/--to-stage to run part of it; progress is in <work>/pipeline_progress.json.
# --limit 30 keeps the agent eval short; raise it for the final choice.
!python -m lifemate.train.llm_pipeline --config configs/train/llm_v1.toml --env colab --limit 30

In [ ]:
!cat artifacts/models/registry.csv
!find /content/drive/MyDrive/zac2026/runs/train -maxdepth 3 -name "metrics.jsonl" -o -maxdepth 3 -name "train_config.json" | head

In [ ]:
# Nothing to do: the work dir is on Drive. Copy the merged model to models/<name>/ when registering it.